# Pipeline & GridSearchCV

Preprocessing dan model disatukan dalam `Pipeline`, lalu hyperparameter dicari dengan `GridSearchCV`.

## 1. Muat Data

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv('../data/Transactions.csv')
df['hour'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.hour
df['is_weekend'] = (pd.to_datetime(df['Date']).dt.dayofweek >= 5).astype(int)

X = df[['TotalItem', 'TotalAmount', 'hour', 'is_weekend', 'Outlet']]
y = df['PaymentMethod']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X.shape

C:\Users\Radit\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Radit\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


(53820, 5)

## 2. Susun Pipeline

In [2]:
num_cols = ['TotalItem', 'TotalAmount', 'hour', 'is_weekend']
cat_cols = ['Outlet']

prep = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first'), cat_cols)])

pipe = Pipeline([
    ('prep', prep),
    ('model', RandomForestClassifier(random_state=42, n_jobs=-1))])

## 3. GridSearchCV

In [3]:
param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [4, 8, None]}

grid = GridSearchCV(pipe, param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)
print('Parameter terbaik:', grid.best_params_)
print('CV accuracy      :', round(grid.best_score_, 3))

Parameter terbaik: {'model__max_depth': 4, 'model__n_estimators': 50}
CV accuracy      : 0.828


## 4. Evaluasi pada Test Set

In [4]:
pred = grid.predict(X_test)
print('Akurasi test:', round(accuracy_score(y_test, pred), 3))

Akurasi test: 0.825
